In [1]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from captum.attr import IntegratedGradients
import pandas as pd
from tqdm import tqdm
import joblib

c:\Users\86155\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# 加载模型和 tokenizer
model_path = "./bert_model/"
model = BertForSequenceClassification.from_pretrained(model_path)
model.load_state_dict(torch.load("best_model.pt", map_location="cpu"))
model.eval()
model.to(device)

tokenizer = BertTokenizer.from_pretrained(model_path)
test_texts = joblib.load("test_texts.pkl")

# 定义 Captum 所需的 forward function
def forward_func(inputs_embeds, attention_mask):
    return model(inputs_embeds=inputs_embeds, attention_mask=attention_mask).logits

ig = IntegratedGradients(forward_func)

# 合并 token 和 attribution 分数的函数
def merge_wordpiece_tokens_with_scores(tokens, scores):
    merged_tokens = []
    merged_scores = []
    current_token = ""
    current_scores = []

    for token, score in zip(tokens, scores):
        if token.startswith("##"):
            current_token += token[2:]
            current_scores.append(score)
        else:
            if current_token:
                merged_tokens.append(current_token)
                merged_scores.append(sum(current_scores) / len(current_scores))
            current_token = token
            current_scores = [score]

    if current_token:
        merged_tokens.append(current_token)
        merged_scores.append(sum(current_scores) / len(current_scores))

    return merged_tokens, merged_scores

# 遍历每条评论，提取所有 token 和分数
token_score_data = []

for text in tqdm(test_texts, desc="Computing Token Attributions"):  # 先处理前 100 条样本用于演示
    encoding = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    input_embed = model.bert.embeddings(input_ids)

    with torch.no_grad():
        logits = model(inputs_embeds=input_embed, attention_mask=attention_mask).logits
        predicted_class = torch.argmax(logits, dim=-1).item()
        label = "positive" if predicted_class == 1 else "negative"

    attributions = ig.attribute(inputs=input_embed,
                                 additional_forward_args=(attention_mask,),
                                 return_convergence_delta=False,
                                 target=predicted_class)

    scores = attributions.norm(p=2, dim=-1).squeeze(0).tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    merged_tokens, merged_scores = merge_wordpiece_tokens_with_scores(tokens, scores)

    for token, score in zip(merged_tokens, merged_scores):
        token_score_data.append({
            "text": text,
            "token": token,
            "attribution_score": score,
            "label": label
        })

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./bert_model/ and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3409756/1175491774.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We rec

In [20]:
# 转为 DataFrame 返回
df_tokens = pd.DataFrame(token_score_data)
print(df_tokens.head())

                                                text         token  \
0  Solicitei cancelamento da compra Prazo de entr...         [CLS]   
1  Solicitei cancelamento da compra Prazo de entr...     Solicitei   
2  Solicitei cancelamento da compra Prazo de entr...  cancelamento   
3  Solicitei cancelamento da compra Prazo de entr...            da   
4  Solicitei cancelamento da compra Prazo de entr...        compra   

   attribution_score     label  
0           0.039490  negative  
1           0.120545  negative  
2           0.180782  negative  
3           0.078450  negative  
4           0.120947  negative  


In [22]:
# 将之前提取的 token-score 数据（df_tokens）直接输出为 CSV 和 HTML 高亮格式

# 保存为 CSV 文件
df_tokens.to_csv("token_attributions_full.csv", index=False, encoding="utf-8-sig")
df_tokens = df_tokens[~df_tokens["token"].isin(["[PAD]", "[CLS]", "[SEP]"])]
# 生成 HTML 高亮
from collections import defaultdict
import html

# 按文本分组
grouped = defaultdict(list)
for _, row in df_tokens.iterrows():
    grouped[row["text"]].append((row["token"], row["attribution_score"], row["label"]))

# 渲染 HTML
def get_color(label):
    return "#ffcccc" if label == "negative" else "#b6fcb6"

# 渲染 HTML token
def render_token_html(tokens, scores, label, threshold=0.2):
    html_tokens = []
    color = get_color(label)
    for token, score in zip(tokens, scores):
        clean = token.replace("##", "")
        clean = html.escape(clean)
        if score >= threshold:
            html_tokens.append(f"<mark style='background-color:{color}'>{clean}</mark>")
        else:
            html_tokens.append(clean)
    return " ".join(html_tokens)

# 构建 HTML 页面内容
html_lines = []
for text, token_group in grouped.items():
    tokens = [t for t, s, l in token_group]
    scores = [s for t, s, l in token_group]
    label = token_group[0][2]
    highlighted = render_token_html(tokens, scores, label)
    html_lines.append(f"<p><b>[{label.upper()}]</b> {highlighted}</p>")

# 写入 HTML 文件
with open("captum_token_explanations.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><style>body{font-family:Arial;}</style></head><body>\n")
    f.write("<h2>情感词高亮可视化</h2>\n")
    for line in html_lines:
        f.write(line + "\n")
    f.write("</body></html>")

"✅ 已生成文件：token_attributions_full.csv 和 captum_token_explanations.html，可用于统计与可视化展示。"


'✅ 已生成文件：token_attributions_full.csv 和 captum_token_explanations.html，可用于统计与可视化展示。'